<a href="https://colab.research.google.com/github/Caffeinboy/epilepsy_detection/blob/main/epilepsy_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install the kagglehub library with pandas support
!pip install kagglehub[pandas-datasets]

import kagglehub
from kagglehub import KaggleDatasetAdapter

# 2. Set the path to the specific file inside the dataset
file_path = "Epileptic Seizure Recognition.csv"

# 3. Download and load the data directly into a dataframe (df)
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "harunshimanto/epileptic-seizure-recognition",
  file_path
)

# 4. Verify it worked
df.head()

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Convert to binary classification: 1 (Seizure) vs 0 (Normal)
df['seizure'] = df['y'].apply(lambda x: 1 if x == 1 else 0)

# 1. Drop the target columns first
X_df = df.drop(['y', 'seizure'], axis=1)

# 2. Keep ONLY numeric columns (this automatically filters out the string ID column)
X = X_df.select_dtypes(include=[np.number]).values
y = df['seizure'].values

# Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the signal data (mean = 0, variance = 1)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Reshape for 1D CNN: (Samples, 178 time steps, 1 feature)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"Training data shape: {X_train.shape}")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, Dropout

model = Sequential([
    Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=(178, 1)),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=16, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(32, activation='relu'),
    Dropout(0.5), # Prevents overfitting
    Dense(1, activation='sigmoid') # Binary output: 0 (Normal) or 1 (Seizure)
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
print("Starting training...")
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test))

# Test the final accuracy
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Model Accuracy on unseen data: {accuracy * 100:.2f}%")

In [ ]:
import tensorflow as tf

# Initialize the converter
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Apply default optimizations (Quantization) to shrink the model size
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Convert the model
tflite_model = converter.convert()

# Save the compressed .tflite file
with open('epilepsy_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("Model compressed and saved as epilepsy_model.tflite")

In [ ]:
!apt-get update && apt-get install -y xxd
!xxd -i epilepsy_model.tflite > epilepsy_model.h
print("Successfully converted to C header file: epilepsy_model.h")

In [ ]:
from google.colab import files
files.download('epilepsy_model.h')